# Asset extraction from the DesignSafe Eaton Fire orthomosaics with `rAPIdtools`

The RAPID Facility flew the Eaton Fire burn area in February and March 2025 and published the RGB orthomosaics on DesignSafe as [PRJ-5815](https://doi.org/10.17603/ds2-wsvx-7s97). The tiles are 4.7 to 20 GB each at 1.9 cm per pixel, so this notebook never copies one. It reads the tile where it already sits, clips a 200 m patch resampled to 5 cm from the tile's overviews, and runs the rapidtools asset pipeline on the patch: SAM 3 detection, image crops, geometric regularization, and an optional Gemini damage assessment.

Where it runs:

- **DesignSafe JupyterHub.** Published data is mounted at `~/NHERI-Published`, so the clip reads a few megabytes out of the tile. Pick a GPU-backed Jupyter HPC session for SAM 3.
- **As a Tapis job.** The `rapidtools-ls6` app in `deploy/tapis/` stages the tile, runs this notebook with [papermill](https://papermill.readthedocs.io/), and archives the outputs. Parameters arrive through the cell tagged `parameters` below.
- **Anywhere else.** `resolve_published_file` downloads with dapi, but refuses files above `max_download_gb`. Every Eaton tile is above the default cap on purpose: tapipy buffers a download in memory before dapi writes it, and the Tapis content endpoint does not serve byte ranges, so large tiles should be read on DesignSafe.

Cite the data as: Allstadt, K. et al. (2025). Eaton Fire burn area, Mission V3. DesignSafe-CI. https://doi.org/10.17603/ds2-wsvx-7s97

In [ ]:
# Parameters. papermill overrides these; edit them by hand when running interactively.
import os

PROJECT_ID = 'PRJ-5815'
IMAGERY_PATH = (
    'Mission--eaton-fire-burn-area--V3/data/'
    'Geoscience-collection--eaton-fire-processed-data-february-march-2025/data/Imagery'
)
TILE = 'Eaton_Trinity38_RGBortho_20250214.tiff'  # 4.7 GB, the smallest tile
RASTER_PATH = None  # a Tapis job passes the staged tile here and skips resolution
PATCH_SIZE_M = 200  # side of the square patch, meters
PATCH_RESOLUTION_M = 0.05  # resample the patch to this pixel size; the tiles are 0.019 m
PATCH_CENTER = None  # (lon, lat) in WGS84; None uses the tile center

TARGET_ASSET = 'building'  # free-form text prompt for SAM 3
APPRX_ASSET_SIZE = 50  # meters, sets the SAM 3 patch size
DETECT_IN_RECON_IMAGERY = True  # False fetches Bing imagery for detection (needs internet)
RUN_DAMAGE_ANALYSIS = False  # True sends crops to Gemini (needs GOOGLE_API_KEY)
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', '')

OUTPUT_DIR = 'output'
UPLOAD_TO = None  # e.g. 'tapis://designsafe.storage.default/<user>/eaton-assets' to copy outputs to My Data
STEPS = 'all'  # 'patch' stops after the clip, which is enough to test data access

### Setup

In [ ]:
import json
import platform
import time
from pathlib import Path

import pandas as pd
import rapidtools
from rapidtools import BoundingBox, configure_logging
from rapidtools.data_sources.designsafe import (
    clip_raster,
    list_published_files,
    published_uri,
    resolve_published_file,
)

configure_logging()
started = time.time()
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
STEM = Path(RASTER_PATH or TILE).stem
IMAGERY_URI = published_uri(PROJECT_ID, IMAGERY_PATH)
TILE_URI = f'{IMAGERY_URI}/{TILE}'
print(f'rapidtools {rapidtools.__version__} on {platform.node()}')
print(TILE_URI if RASTER_PATH is None else f'Staged tile: {RASTER_PATH}')

### 1. The imagery

List the tiles in the publication with their sizes. This is the only call that needs dapi credentials when running on JupyterHub; a Tapis job already has the tile and skips it.

In [ ]:
if RASTER_PATH is None:
    tiles = pd.DataFrame(list_published_files(IMAGERY_URI))
    display(tiles[['name', 'size_gb']])
    assert TILE in set(tiles['name']), f'{TILE} is not in {IMAGERY_URI}'

Resolve the tile to a readable path and clip the patch. On JupyterHub the resolver finds the file under `~/NHERI-Published`; nothing is downloaded. The clip reads only the window, from the tile's overviews when `PATCH_RESOLUTION_M` is coarser than the source. At full resolution a 200 m window is a 10,000 pixel square and takes minutes; at 5 cm it is 4,000 pixels and takes seconds.

In [ ]:
source = Path(RASTER_PATH) if RASTER_PATH else resolve_published_file(TILE_URI)
raster_path = clip_raster(
    source,
    out / f'{STEM}_patch.tiff',
    center=tuple(PATCH_CENTER) if PATCH_CENTER else None,
    size_m=PATCH_SIZE_M,
    resolution_m=PATCH_RESOLUTION_M,
)
region_of_interest = BoundingBox.from_raster(raster_path)
print(raster_path, f'{raster_path.stat().st_size / 1e6:.1f} MB')
region_of_interest

In [ ]:
import matplotlib.pyplot as plt
import rasterio

with rasterio.open(raster_path) as src:
    scale = max(1, max(src.width, src.height) // 1024)
    preview = src.read(
        [1, 2, 3],
        out_shape=(3, src.height // scale, src.width // scale),
    ).transpose(1, 2, 0)
plt.figure(figsize=(7, 7))
plt.imshow(preview)
plt.title(f'{STEM} patch, {PATCH_SIZE_M} m at {PATCH_RESOLUTION_M} m')
plt.axis('off')
plt.show()

### 2. Extract preliminary asset geometries with SAM 3

With `DETECT_IN_RECON_IMAGERY=True` SAM 3 runs on the RAPID patch itself. Set it to `False` to detect on Bing imagery of the same footprint instead, which needs internet access and a pre-event basemap.

In [ ]:
if STEPS == 'patch':
    print('STEPS=patch: stopping after the clip.')
else:
    from rapidtools import SAM3OrthoFeatureExtractor

    sam_extractor = SAM3OrthoFeatureExtractor(
        prompt=TARGET_ASSET,
        patch_size=APPRX_ASSET_SIZE,
        unit='meters',
        overlap_ratio=0.25,
        batch_size=4,
        threshold=0.50,
        mask_threshold=0.40,
    )
    if DETECT_IN_RECON_IMAGERY:
        detection_raster = raster_path
    else:
        from rapidtools import BingOrthomosaicExtractor

        detection_raster = BingOrthomosaicExtractor(max_workers=10)(
            region=region_of_interest, output_path=str(out / f'{STEM}_bing.tiff')
        )
    prelim_collection = sam_extractor(detection_raster)
    prelim_path = out / f'{STEM}_assets_preliminary.geojson'
    _ = prelim_collection.to_geojson(str(prelim_path))
    print(f'{len(prelim_collection)} preliminary assets')

### 3. Crop each asset with context and regularize the geometry

In [ ]:
if STEPS != 'patch':
    from rapidtools import (
        AerialImageryExtractor,
        BuildingRegularizer,
        RoadwayRegularizer,
    )

    crops = AerialImageryExtractor(
        dataset=detection_raster,
        save_directory=str(out / 'asset_crops'),
        buffer_asset='20 m',
        force_square_image=True,
    )
    assets_with_images = crops(prelim_collection)

    if 'building' in TARGET_ASSET:
        regularizer = BuildingRegularizer(batch_size=8)
    elif 'road' in TARGET_ASSET:
        regularizer = RoadwayRegularizer()
    else:
        regularizer = None
    final_assets = regularizer(assets_with_images) if regularizer else assets_with_images
    final_path = out / f'{STEM}_assets_final.geojson'
    _ = final_assets.to_geojson(str(final_path))
    print(f'{len(final_assets)} assets written to {final_path}')

### 4. Post-disaster condition assessment (optional)

Grades each asset crop from the RAPID patch on the Cal Fire / CHS scale with Gemini. Needs `GOOGLE_API_KEY` and internet access.

In [ ]:
if STEPS != 'patch' and RUN_DAMAGE_ANALYSIS and GOOGLE_API_KEY:
    from rapidtools import GeminiAssetAnalyzer, Pipeline, download_dataset

    [prompt_path] = download_dataset('aerial_chs_prompts', output_dir=str(out))
    pipeline = Pipeline()
    pipeline.add_step(
        AerialImageryExtractor(
            dataset=raster_path,
            save_directory=str(out / 'overlaid_imagery'),
            overlay_asset_outline=True,
            image_prefix=STEM,
            keep_multiple_copies=True,
        )
    )
    pipeline.add_step(
        GeminiAssetAnalyzer(
            api_key=GOOGLE_API_KEY,
            prompt=Path(prompt_path).read_text().strip(),
        )
    )
    inferred = pipeline.run(final_assets).filter_empty()
    inferred_path = out / f'{STEM}_assets_inferred.geojson'
    _ = inferred.to_geojson(str(inferred_path), ignore_properties=['image_assets'])
    print(f'{len(inferred)} graded assets written to {inferred_path}')
elif STEPS != 'patch':
    print('Damage assessment skipped: set RUN_DAMAGE_ANALYSIS=True and GOOGLE_API_KEY.')

### 5. Record what ran

A manifest next to the outputs pins the tile, the patch window, every parameter, and the software version, so the run can be reproduced without this notebook's history.

In [ ]:
manifest = {
    'tile_uri': TILE_URI if RASTER_PATH is None else None,
    'raster_path': str(source),
    'patch': {'path': str(raster_path), 'bounds_wgs84': region_of_interest.bounds, 'size_m': PATCH_SIZE_M, 'resolution_m': PATCH_RESOLUTION_M, 'center': PATCH_CENTER},
    'parameters': {
        'target_asset': TARGET_ASSET,
        'approx_asset_size_m': APPRX_ASSET_SIZE,
        'detect_in_recon_imagery': DETECT_IN_RECON_IMAGERY,
        'run_damage_analysis': RUN_DAMAGE_ANALYSIS,
        'steps': STEPS,
    },
    'software': {'rapidtools': rapidtools.__version__, 'python': platform.python_version()},
    'job_uuid': os.environ.get('_tapisJobUUID'),
    'outputs': sorted(str(p.relative_to(out)) for p in out.rglob('*') if p.is_file()),
    'seconds': round(time.time() - started, 1),
}
(out / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest, indent=2, default=str))

### 6. Save to My Data (optional)

A Tapis job archives `output/` on its own. On JupyterHub or a laptop, set `UPLOAD_TO` to a `tapis://designsafe.storage.default/<username>/...` folder to copy the outputs there with dapi.

In [ ]:
if UPLOAD_TO:
    from dapi import DSClient

    ds = DSClient()
    for path in sorted(p for p in out.rglob('*') if p.is_file() and p.suffix in {'.geojson', '.json', '.tiff'}):
        ds.files.upload(str(path), f'{UPLOAD_TO.rstrip("/")}/{path.relative_to(out)}')
    print(f'Uploaded outputs to {UPLOAD_TO}')